# Cria do Tatame — Colab/Drive Pipeline v1

Lote vertical seguro: monta o Drive privado, fixa a revisão de um modelo do Hugging Face, gera a fila canônica e empacota especificações em `assets/candidatos`. Este notebook **não promove assets**, **não declara arte final** e não altera o runtime Godot.


In [ ]:
import json
import os
import re
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

REPO_URL = 'https://github.com/ringuemkt-rgb/cria-do-tatame.git'
REPO_REF = os.environ.get('CRIA_GIT_REF', 'main')
if not re.fullmatch(r'[A-Za-z0-9._/-]+', REPO_REF) or '..' in REPO_REF:
    raise ValueError('CRIA_GIT_REF contains unsafe characters')
CHECKOUT = Path('/content/cria-do-tatame')
DRIVE_BASE = Path('/content/drive/MyDrive/CriaDoTatame')
MODEL_ID = 'ByteDance/AnimateDiff-Lightning'
BATCH_KIND = 'paired_technique_animation'
BATCH_TARGET = 'baiana_single_leg'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

required = [
    'models', 'cache', 'refs', 'motions', 'assets/candidatos',
    'assets/aprovados', 'audio/sfx', 'audio/vozes', 'colab_logs', 'manifest',
]
missing = [name for name in required if not (DRIVE_BASE / name).is_dir()]
if missing:
    raise FileNotFoundError(f'Drive hierarchy is incomplete: {missing}')
print({'drive_ready': True, 'root': str(DRIVE_BASE)})


In [ ]:
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print({'cuda_available': torch.cuda.is_available(), 'gpu': gpu})
if not torch.cuda.is_available():
    print('Aviso: empacotamento funciona em CPU; geração visual deve aguardar uma sessão com GPU.')


In [ ]:
if not (CHECKOUT / '.git').is_dir():
    subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', REPO_URL, str(CHECKOUT)], check=True)
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', REPO_REF], cwd=CHECKOUT, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=CHECKOUT, check=True)
commit = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], cwd=CHECKOUT, check=True, capture_output=True, text=True
).stdout.strip()
print({'checkout': str(CHECKOUT), 'commit': commit})


In [ ]:
requirements = CHECKOUT / 'tools/ai_asset_pipeline/cloud/requirements-colab.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)


In [ ]:
queue_path = CHECKOUT / 'tools/ai_asset_pipeline/generated_queue/production_queue_v02.jsonl'
subprocess.run(
    [sys.executable, 'tools/ai_asset_pipeline/build_production_queue_v02.py',
     '--kind', 'techniques', '--output', str(queue_path)],
    cwd=CHECKOUT, check=True,
)
print({'queue': str(queue_path), 'bytes': queue_path.stat().st_size})


In [ ]:
run_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
snapshot_path = DRIVE_BASE / 'manifest' / f'hf_model_snapshot_{commit[:12]}_{run_stamp}.json'
subprocess.run(
    [sys.executable, 'tools/ai_asset_pipeline/cloud/resolve_hf_models.py',
     '--model', MODEL_ID, '--allow-research', '--output', str(snapshot_path)],
    cwd=CHECKOUT, check=True,
)
snapshot = json.loads(snapshot_path.read_text(encoding='utf-8'))
print({'model': MODEL_ID, 'revision': snapshot['models'][0]['resolved_revision']})


In [ ]:
candidate_dir = DRIVE_BASE / 'assets/candidatos'
result = subprocess.run(
    [sys.executable, 'tools/ai_asset_pipeline/cloud/prepare_batch.py',
     '--queue', str(queue_path), '--model-registry', str(snapshot_path),
     '--output-dir', str(candidate_dir), '--kind', BATCH_KIND,
     '--target', BATCH_TARGET, '--limit', '1', '--source-commit', commit],
    cwd=CHECKOUT, check=True, capture_output=True, text=True,
)
receipt = json.loads(result.stdout)
print(receipt)


In [ ]:
log_path = DRIVE_BASE / 'colab_logs' / f"batch_{receipt['batch_id']}.json"
log_path.write_text(json.dumps({
    'schema_version': 1,
    'source_commit': commit,
    'model_snapshot': str(snapshot_path.relative_to(DRIVE_BASE)),
    'receipt': receipt,
    'next_gate': 'human_generation_and_QA',
}, ensure_ascii=False, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print({'completed': True, 'candidate_bundle': receipt['bundle'], 'log': str(log_path)})


## Estado ao final

O ZIP gerado contém tarefas, checksums, commit do Git e revisão imutável do modelo. Ele permanece candidato. Renderização, validação biomecânica, limpeza de pixel art, QA visual, aprovação humana e integração no Godot são gates posteriores.
